In [1]:
import torch
import math
import os
import random
torch.manual_seed(35)
random.seed(35)

In [2]:
# def cot_trace(a, b):
#     da = [int(c) for c in str(a)][::-1]
#     db = [int(c) for c in str(b)][::-1]
#     n = max(len(da), len(db))
#     da += [0] * (n - len(da))
#     db += [0] * (n - len(db))
#     steps, carry = [], 0
#     for i in range(n):
#         s = da[i] + db[i] + carry
#         new_carry = s // 10
#         if carry:
#             steps.append(f'{da[i]}+{db[i]}+{carry}={s}c{new_carry}')
#         else:
#             steps.append(f'{da[i]}+{db[i]}={s}c{new_carry}')
#         carry = new_carry
#     return ','.join(steps)

# def make_reversed_line(a, b):
#     sa = str(a)[::-1]                    # units first
#     sb = str(b)[::-1]
#     sr = str(a + b)[::-1]                # answer, units first

#     n = max(len(sa), len(sb))
#     sa = sa.ljust(n, '0')                # pad shorter operand with zeros on the right
#     sb = sb.ljust(n, '0')

#     steps, carry = [], 0
#     for i in range(n):
#         da, db = int(sa[i]), int(sb[i])
#         s = da + db + carry
#         digit = s % 10                   # single digit written this column
#         new_carry = s // 10
#         if carry:
#             steps.append(f'{da}+{db}+{carry}={digit}c{new_carry}')
#         else:
#             steps.append(f'{da}+{db}={digit}c{new_carry}')
#         carry = new_carry
#     if carry:
#         steps.append(f'c{carry}')        # final carry becomes final digit of answer

#     return f'{sa}+{sb}=|{",".join(steps)}|{sr}'

# MAX_DIGITS = 2
# lines = []
# for _ in range(2000):
#     a = random.randint(0, 10**MAX_DIGITS - 1)
#     b = random.randint(0, 10**MAX_DIGITS - 1)
#     lines.append(make_reversed_line(a, b))

# corpus = '\n'.join(lines)
# print('Preview (remember: leftmost digit is the units digit):')
# print(corpus[:400])

# # Character-level vocab
# chars = sorted(set(corpus))
# vocab = chars
# char_to_index = {c: i for i, c in enumerate(vocab)}
# index_to_char = {i: c for c, i in char_to_index.items()}
# vocab_size = len(vocab)
# vocab_size

In [3]:
# Build the corpus: addition problems with a chain-of-thought.
# Example line:  27+45=|7+5=12c1,2+4+1=7c0|72
#   - left of first |  : the problem
#   - between | | : per-column work, right-to-left, with carries
#   - after last | : the final answer

def cot_trace(a, b):
    da = [int(c) for c in str(a)][::-1]
    db = [int(c) for c in str(b)][::-1]
    n = max(len(da), len(db))
    da += [0] * (n - len(da))
    db += [0] * (n - len(db))
    steps, carry = [], 0
    for i in range(n):
        s = da[i] + db[i] + carry
        new_carry = s // 10
        if carry:
            steps.append(f'{da[i]}+{db[i]}+{carry}={s}c{new_carry}')
        else:
            steps.append(f'{da[i]}+{db[i]}={s}c{new_carry}')
        carry = new_carry
    return ','.join(steps)

MAX_DIGITS = 2
lines = []
for _ in range(2000):
    a = random.randint(0, 10**MAX_DIGITS - 1)
    b = random.randint(0, 10**MAX_DIGITS - 1)
    lines.append(f'{a}+{b}=|{cot_trace(a, b)}|{a+b}')

corpus = '\n'.join(lines)
print(corpus[:300])  # preview

# Character-level vocab (each character is a token). Digits and symbols each become their own id.
chars = sorted(set(corpus))
vocab = chars
char_to_index = {c: i for i, c in enumerate(vocab)}
index_to_char = {i: c for c, i in char_to_index.items()}
vocab_size = len(vocab)
vocab_size

70+42=|0+2=2c0,7+4=11c1|112
96+16=|6+6=12c1,9+1+1=11c1|112
95+43=|5+3=8c0,9+4=13c1|138
19+36=|9+6=15c1,1+3+1=5c0|55
55+32=|5+2=7c0,5+3=8c0|87
72+7=|2+7=9c0,7+0=7c0|79
95+67=|5+7=12c1,9+6+1=16c1|162
96+35=|6+5=11c1,9+3+1=13c1|131
83+46=|3+6=9c0,8+4=12c1|129
91+74=|1+4=5c0,9+7=16c1|165
96+12=|6+2=8c0,


16

In [4]:
print(corpus[:500])

70+42=|0+2=2c0,7+4=11c1|112
96+16=|6+6=12c1,9+1+1=11c1|112
95+43=|5+3=8c0,9+4=13c1|138
19+36=|9+6=15c1,1+3+1=5c0|55
55+32=|5+2=7c0,5+3=8c0|87
72+7=|2+7=9c0,7+0=7c0|79
95+67=|5+7=12c1,9+6+1=16c1|162
96+35=|6+5=11c1,9+3+1=13c1|131
83+46=|3+6=9c0,8+4=12c1|129
91+74=|1+4=5c0,9+7=16c1|165
96+12=|6+2=8c0,9+1=10c1|108
46+1=|6+1=7c0,4+0=4c0|47
65+1=|5+1=6c0,6+0=6c0|66
12+92=|2+2=4c0,1+9=10c1|104
63+47=|3+7=10c1,6+4+1=11c1|110
93+3=|3+3=6c0,9+0=9c0|96
43+86=|3+6=9c0,4+8=12c1|129
63+6=|3+6=9c0,6+0=6c0|69



In [5]:
embed_dim = 128
number_of_heads = 8
head_size = embed_dim // number_of_heads
ffn_hidden = 512

In [6]:
# all embeddings
embedding_table = torch.nn.Parameter(torch.randn(vocab_size, embed_dim))

# decoder self attention weights
dec_wq = torch.nn.Parameter(torch.randn(embed_dim, embed_dim))
dec_wk = torch.nn.Parameter(torch.randn(embed_dim, embed_dim))
dec_wv = torch.nn.Parameter(torch.randn(embed_dim, embed_dim))
dec_wo = torch.nn.Parameter(torch.randn(embed_dim, embed_dim))

# decoder FFN
dec_ff1_w = torch.nn.Parameter(torch.randn(embed_dim, ffn_hidden))
dec_ff1_b = torch.nn.Parameter(torch.zeros(ffn_hidden))
dec_ff2_w = torch.nn.Parameter(torch.randn(ffn_hidden, embed_dim))
dec_ff2_b = torch.nn.Parameter(torch.zeros(embed_dim))

# Logits function weights
out_w = torch.nn.Parameter(torch.randn(embed_dim, vocab_size))
out_b = torch.nn.Parameter(torch.zeros(vocab_size))

# normalization layers
dec_norm1 = torch.nn.LayerNorm(embed_dim)
dec_norm2 = torch.nn.LayerNorm(embed_dim)

all_params = [
    embedding_table,
    dec_wq, dec_wk, dec_wv, dec_wo,
    dec_ff1_w, dec_ff1_b, dec_ff2_w, dec_ff2_b,
    out_w, out_b,
    *dec_norm1.parameters(),
    *dec_norm2.parameters(),
]

optimizer = torch.optim.Adam(all_params, lr=0.001)
loss_fn   = torch.nn.CrossEntropyLoss()

In [7]:
def pos_enc(seq_len):
    pe = torch.zeros(seq_len, embed_dim)
    for pos in range(seq_len):
        for i in range(0, embed_dim, 2):
            pe[pos, i]   = math.sin(pos / 10000 ** (i / embed_dim))
            pe[pos, i+1] = math.cos(pos / 10000 ** (i / embed_dim))
    return pe

In [8]:
training_sequences = []
for line in corpus.strip().split('\n'):
    line = line + '\n'
    ids = [char_to_index[c] for c in line]
    source_token = torch.tensor(ids[:-1])
    target_token = torch.tensor(ids[1:])
    training_sequences.append((source_token, target_token))

print(f'Training sequences: {len(training_sequences)}')
print('First sequence chars:', ''.join(index_to_char[i.item()] for i in training_sequences[0][0]))

Training sequences: 2000
First sequence chars: 70+42=|0+2=2c0,7+4=11c1|112


In [9]:
if os.path.exists('reasoning_weights.pth'):
    checkpoint = torch.load('reasoning_weights.pth')
    embedding_table.data = checkpoint['embedding_table'].data
    dec_wq.data = checkpoint['dec_wq'].data
    dec_wk.data = checkpoint['dec_wk'].data
    dec_wv.data = checkpoint['dec_wv'].data
    dec_wo.data = checkpoint['dec_wo'].data
    dec_ff1_w.data = checkpoint['dec_ff1_w'].data
    dec_ff1_b.data = checkpoint['dec_ff1_b'].data
    dec_ff2_w.data = checkpoint['dec_ff2_w'].data
    dec_ff2_b.data = checkpoint['dec_ff2_b'].data
    out_w.data = checkpoint['out_w'].data
    out_b.data = checkpoint['out_b'].data
    dec_norm1.load_state_dict(checkpoint['dec_norm1'])
    dec_norm2.load_state_dict(checkpoint['dec_norm2'])
    print('Loaded existing weights')
else:
    print('No saved weights, training model from scratch.')

No saved weights, training model from scratch.


In [10]:
def neural_network_decoder(source_tokens):
    sequence_length = len(source_tokens)
    source_embedding = embedding_table[source_tokens] + pos_enc(sequence_length)

    #masked self attention, hides next element in series using '-inf' for prediction
    mask_self_attention = torch.triu(torch.ones(sequence_length, sequence_length), diagonal=1)
    mask_self_attention = mask_self_attention.masked_fill(mask_self_attention ==1, float('-inf'))

    Qd = source_embedding @ dec_wq
    Kd = source_embedding @ dec_wk
    Vd = source_embedding @ dec_wv

    # Splitting Qd,Kd, Vd to make it multi-headed 
    Qd = Qd.view(sequence_length, number_of_heads, head_size).transpose(0, 1)
    Kd = Kd.view(sequence_length, number_of_heads, head_size).transpose(0, 1)
    Vd = Vd.view(sequence_length, number_of_heads, head_size).transpose(0, 1)

    scores_dec = Qd @ Kd.transpose(-2, -1) / math.sqrt(head_size) + mask_self_attention
    attention_dec = torch.softmax(scores_dec, dim=-1)
    out_dec = attention_dec @ Vd 

    # Concatenating multi-head or joining them back
    out_dec = out_dec.transpose(0, 1).contiguous().view(sequence_length, embed_dim)

    #Output Projection Layer
    out_dec = out_dec @ dec_wo

    # Residual Layer1
    residual_layer1 = source_embedding + out_dec

    # Normalization Layer
    normalization_layer1 = dec_norm1(residual_layer1)

    #FFN
    # Linear Layer
    linear_layer1 = normalization_layer1 @ dec_ff1_w + dec_ff1_b
    relu_layer1 = torch.relu(linear_layer1)
    linear_layer2 = relu_layer1 @ dec_ff2_w + dec_ff2_b

    #Residual Layer2
    residual_layer2 = normalization_layer1 + linear_layer2

    # Normalization Layer2
    normalization_layer2 = dec_norm2(residual_layer2)

    #Logits Layer
    logits = normalization_layer2 @ out_w + out_b
    return logits

In [11]:
epochs = 20
for epoch in range(1, epochs + 1):
    total_loss = 0.0

    random.shuffle(training_sequences)
    for source_tokens, target_tokens in training_sequences:

        logits = neural_network_decoder(source_tokens)
        loss = loss_fn(logits, target_tokens)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(all_params, max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
    avg_loss = total_loss / len(training_sequences)
    print('epoch', epoch, 'avg_loss', avg_loss)

epoch 1 avg_loss 2.095718839466572
epoch 2 avg_loss 1.2689047876298427
epoch 3 avg_loss 1.1432282779216767
epoch 4 avg_loss 1.0560973817408086
epoch 5 avg_loss 1.0041283321976662
epoch 6 avg_loss 0.9713460332304239
epoch 7 avg_loss 0.9329176889359951
epoch 8 avg_loss 0.8834911889433861
epoch 9 avg_loss 0.8898825644403696
epoch 10 avg_loss 0.8624329333156348
epoch 11 avg_loss 0.8434226627498865
epoch 12 avg_loss 0.8251789876967669
epoch 13 avg_loss 0.7880835837572813
epoch 14 avg_loss 0.7751131198853254
epoch 15 avg_loss 0.7517535562068224
epoch 16 avg_loss 0.7365864085555076
epoch 17 avg_loss 0.7269548445641995
epoch 18 avg_loss 0.7051384077370166
epoch 19 avg_loss 0.7057402935773134
epoch 20 avg_loss 0.7074768734201788


In [12]:
torch.save({
    'embedding_table': embedding_table,
    'dec_wq': dec_wq, 'dec_wk': dec_wk, 'dec_wv': dec_wv, 'dec_wo': dec_wo,
    'dec_ff1_w': dec_ff1_w, 'dec_ff1_b': dec_ff1_b,
    'dec_ff2_w': dec_ff2_w, 'dec_ff2_b': dec_ff2_b,
    'out_w': out_w, 'out_b': out_b,
    'dec_norm1': dec_norm1.state_dict(),
    'dec_norm2': dec_norm2.state_dict(),
    'char_to_index': char_to_index,
    'index_to_char': index_to_char,
    'vocab_size': vocab_size,
}, 'reasoning_weights.pth')
print('Saved.')

Saved.


In [13]:
def generate(prompt, max_chars=60, temperature=0.0):
    unknown = [c for c in prompt if c not in char_to_index]
    if unknown:
        print(f'Unknown chars: {unknown}')
        return

    generated = list(prompt)

    with torch.no_grad():
        for _ in range(max_chars):
            tokens = torch.tensor([char_to_index[c] for c in generated])
            logits = neural_network_decoder(tokens)
            if temperature == 0.0:
                next_token = logits[-1].argmax().item()
            else:
                probs = torch.softmax(logits[-1] / temperature, dim=-1)
                next_token = torch.multinomial(probs, 1).item()
            next_char = index_to_char[next_token]
            if next_char == '\n':
                break
            generated.append(next_char)

    output = ''.join(generated)
    print(f'{prompt}   ->   {output}')
    return output

# test
generate('27+45=')
generate('9+8=')
generate('50+50=')

27+45=   ->   27+45=|7+2=10c1,7+6+1=10c1|16
9+8=   ->   9+8=|9+8=16c1|1
50+50=   ->   50+50=|0+0=3c0,7+9=10c1|108


'50+50=|0+0=3c0,7+9=10c1|108'

In [14]:
# def generate_raw(prompt, max_chars=60):
#     unknown = [c for c in prompt if c not in char_to_index]
#     if unknown:
#         print(f'Unknown chars: {unknown}')
#         return prompt

#     generated = list(prompt)
#     with torch.no_grad():
#         for _ in range(max_chars):
#             tokens = torch.tensor([char_to_index[c] for c in generated])
#             logits = neural_network_decoder(tokens)
#             next_token = logits[-1].argmax().item()
#             next_char = index_to_char[next_token]
#             if next_char == '\n':
#                 break
#             generated.append(next_char)
#     return ''.join(generated)

# def ask(a, b):
#     # 1. reverse the operands so the model sees units-first
#     sa = str(a)[::-1]
#     sb = str(b)[::-1]
#     n = max(len(sa), len(sb))
#     sa = sa.ljust(n, '0')
#     sb = sb.ljust(n, '0')
#     prompt = f'{sa}+{sb}='

#     # 2. run the model
#     raw = generate_raw(prompt)

#     # 3. everything after the last '|' is the reversed answer; reverse it back to a real number
#     tail = raw.split('|')[-1]
#     digits = ''.join(c for c in tail if c.isdigit())
#     predicted = int(digits[::-1]) if digits else None

#     print(f'{a}+{b} = {predicted}   (truth {a+b})   [raw model output: {raw}]')
#     return predicted

# # test
# ask(27, 45)
# ask(9, 8)
# ask(50, 50)
# ask(83, 30)

In [15]:
vocab_size

16